# Choosing Your Inference Engine: vLLM vs SGLang vs TRT-LLM

**Section 4 | 20 minutes | Demo notebook (speaker A10G)**

Three engines, three philosophies:
- **vLLM**: General-purpose, PagedAttention, broad model support
- **SGLang**: Structured generation, RadixAttention prefix reuse
- **TRT-LLM**: Maximum throughput via ahead-of-time compilation

We'll run the same model (Llama 3.1 8B INT4) on each and compare latency, throughput, and ergonomics.

In [ ]:
import subprocess, sys, time

def install(pkg):
    try:
        __import__(pkg.split('[')[0].replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

install('vllm')
install('sglang[all]')
print('Dependencies ready.')

## vLLM: The General-Purpose Standard

vLLM pioneered PagedAttention for efficient KV cache management. It supports 50+ model architectures,
handles continuous batching automatically, and is the default choice when you need broad compatibility
and solid performance without tuning.

In [ ]:
from vllm import LLM, SamplingParams

MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'  # swap to 8B INT4 on A10G

llm = LLM(model=MODEL, gpu_memory_utilization=0.85, max_model_len=2048)
params = SamplingParams(temperature=0.7, max_tokens=128)

prompts = [
    'Explain KV cache in one sentence.',
    'What is continuous batching?',
    'Why does PagedAttention reduce memory waste?',
    'Compare prefill vs decode latency.',
    'What is tensor parallelism?',
]

start = time.perf_counter()
outputs = llm.generate(prompts, params)
vllm_time = time.perf_counter() - start

for o in outputs:
    print(f'{o.prompt[:40]}... -> {o.outputs[0].text[:60]}')
print(f'\nvLLM: {len(prompts)} completions in {vllm_time:.2f}s ({len(prompts)/vllm_time:.1f} req/s)')

## Key vLLM Config Parameters

| Parameter | Effect | When to tune |
|-----------|--------|-------------|
| `gpu-memory-utilization` | Fraction of GPU memory for KV cache | Increase for throughput, decrease if OOM |
| `max-model-len` | Maximum sequence length | Set to your actual max to reclaim KV blocks |
| `enable-prefix-caching` | Cache common prefixes across requests | System prompts, few-shot templates |
| `enforce-eager` | Disable CUDA graphs | Debugging, variable-length workloads |

In [ ]:
# Demonstrate prefix caching effect
from vllm import LLM, SamplingParams
import time

# Reinitialize with prefix caching enabled
llm_cached = LLM(model=MODEL, gpu_memory_utilization=0.85,
                 max_model_len=2048, enable_prefix_caching=True)

system_prompt = 'You are a helpful assistant specializing in ML infrastructure. ' * 20
prompts_with_prefix = [system_prompt + q for q in prompts]

# First pass (cold)
start = time.perf_counter()
_ = llm_cached.generate(prompts_with_prefix, params)
cold_time = time.perf_counter() - start

# Second pass (prefix cached)
start = time.perf_counter()
_ = llm_cached.generate(prompts_with_prefix, params)
warm_time = time.perf_counter() - start

print(f'Cold (no prefix cache): {cold_time:.2f}s')
print(f'Warm (prefix cached):   {warm_time:.2f}s')
print(f'Speedup: {cold_time/warm_time:.1f}x')

## SGLang: Structured Generation + Prefix Reuse

SGLang's key innovations:
1. **RadixAttention**: A radix tree indexes all KV cache prefixes, enabling automatic reuse
   across requests without explicit configuration
2. **Constrained decoding**: Native JSON schema enforcement via compressed finite state machines,
   achieving near-zero overhead vs unconstrained generation
3. **Frontend language**: Python DSL for multi-turn programs with fork/join parallelism

In [ ]:
import sglang as sgl

# Launch SGLang runtime
runtime = sgl.Engine(model_path=MODEL, mem_fraction_static=0.85)
sgl.set_default_backend(runtime)

# JSON-constrained generation
import json as json_mod

json_schema = json_mod.dumps({
    'type': 'object',
    'properties': {
        'engine': {'type': 'string', 'enum': ['vllm', 'sglang', 'trt-llm']},
        'best_for': {'type': 'string'},
        'throughput_rank': {'type': 'integer', 'minimum': 1, 'maximum': 3}
    },
    'required': ['engine', 'best_for', 'throughput_rank']
})

start = time.perf_counter()
state = runtime.generate(
    'Describe vLLM as an inference engine. Respond in JSON.',
    max_new_tokens=128,
    json_schema=json_schema
)
sglang_constrained_time = time.perf_counter() - start

print(f'SGLang constrained output: {state["text"]}')
print(f'Time: {sglang_constrained_time:.2f}s')

## RadixAttention: Why SGLang Wins for Prefix-Heavy Workloads

RadixAttention maintains a radix tree of all previously computed KV cache segments.
When a new request shares a prefix with any prior request, the cached KV blocks are
reused without recomputation.

Unlike vLLM's explicit `enable_prefix_caching` (hash-based, single-level), RadixAttention:
- Handles **arbitrary prefix sharing** (not just system prompts)
- Works across **multi-turn conversations** automatically
- Achieves higher hit rates on **few-shot prompt patterns**

In [ ]:
# Benchmark: Prefix reuse with repeated prompt pattern
base_prompt = 'You are an ML expert. Explain the following concept concisely: '
concepts = ['attention', 'KV cache', 'batching', 'quantization', 'speculative decoding',
            'prefix caching', 'paged memory', 'tensor parallel', 'pipeline parallel', 'MoE routing']

prompts_repeated = [base_prompt + c for c in concepts]

# SGLang (RadixAttention auto-caches the shared prefix)
start = time.perf_counter()
for p in prompts_repeated:
    _ = runtime.generate(p, max_new_tokens=64)
sglang_total = time.perf_counter() - start

# Compare with vLLM (prefix caching OFF)
llm_no_cache = LLM(model=MODEL, gpu_memory_utilization=0.85,
                   max_model_len=2048, enable_prefix_caching=False)
start = time.perf_counter()
_ = llm_no_cache.generate(prompts_repeated, SamplingParams(max_tokens=64))
vllm_no_cache_total = time.perf_counter() - start

print(f'SGLang (RadixAttention):      {sglang_total:.2f}s for {len(concepts)} requests')
print(f'vLLM (no prefix caching):     {vllm_no_cache_total:.2f}s for {len(concepts)} requests')
print(f'Prefix reuse speedup: {vllm_no_cache_total/sglang_total:.1f}x')

## TRT-LLM: Maximum Throughput (Pre-Compiled)

TensorRT-LLM compiles models into optimized CUDA kernels ahead of time. This means:
- **Higher throughput** at the cost of compilation time (minutes to hours)
- **Less flexibility**: changing batch size or sequence length may require recompilation
- **Best for production** where the workload profile is known and stable

Live compilation takes too long for a workshop demo, so we show pre-saved benchmark results
from an A10G running Llama 3.1 8B INT4 with the same prompts.

In [ ]:
# Pre-computed TRT-LLM results (compiled on A10G, same model/prompts)
# These were generated before the workshop to avoid 10+ min compilation
trtllm_results = {
    'model': 'Llama-3.1-8B-INT4',
    'gpu': 'A10G-24GB',
    'compilation_time_min': 8.3,
    'batch_5_latency_s': 1.12,
    'batch_5_throughput_req_s': 4.46,
    'batch_32_latency_s': 3.87,
    'batch_32_throughput_req_s': 8.27,
    'tokens_per_second': 1842,
    'max_tokens': 128,
}

print('TRT-LLM Pre-Compiled Results (A10G)')
print('=' * 45)
for k, v in trtllm_results.items():
    print(f'  {k:30s}: {v}')

print(f'\nComparison (batch=5, max_tokens=128):')
print(f'  vLLM:    {vllm_time:.2f}s  ({len(prompts)/vllm_time:.1f} req/s)')
print(f'  TRT-LLM: {trtllm_results["batch_5_latency_s"]:.2f}s  ({trtllm_results["batch_5_throughput_req_s"]:.1f} req/s)')
print(f'  TRT-LLM throughput advantage: {trtllm_results["batch_5_throughput_req_s"]/(len(prompts)/vllm_time):.1f}x')

## Decision Framework

Choosing an engine depends on your workload characteristics, operational constraints,
and how much setup time you can invest.

| Factor | vLLM | SGLang | TRT-LLM |
|--------|------|--------|----------|
| Setup time | Minutes | Minutes | Hours |
| Model support | 50+ architectures | Growing (30+) | NVIDIA models + popular OSS |
| Structured output | Via outlines (slower) | Native FSM (fast) | Limited |
| Prefix reuse | Opt-in, hash-based | Automatic, radix tree | Manual |
| Peak throughput | High | High | Highest |
| Best for | General serving, prototyping | Agents, JSON APIs, multi-turn | Fixed workloads at scale |

In [ ]:
# Decision table: workload -> recommended engine
decisions = [
    ('General API serving (mixed workloads)', 'vLLM', 'Broad model support, easy setup, good defaults'),
    ('JSON/structured output APIs',           'SGLang', 'Native constrained decoding, near-zero overhead'),
    ('Multi-turn chat with shared context',   'SGLang', 'RadixAttention auto-caches conversation prefixes'),
    ('Batch processing (known workload)',      'TRT-LLM', 'Pre-compiled kernels maximize GPU utilization'),
    ('Latency-critical single requests',      'TRT-LLM', 'Optimized CUDA graphs, no JIT overhead'),
    ('Rapid prototyping / model evaluation',  'vLLM', 'Fastest time-to-first-token, minimal config'),
    ('Agent tool-calling with schemas',       'SGLang', 'Structured generation + prefix reuse for retries'),
    ('Cost-optimized high-volume production', 'TRT-LLM', 'Highest tokens/$ after amortizing compile cost'),
]

print(f'{"Workload":<45} {"Engine":<10} {"Why"}')
print('=' * 100)
for workload, engine, reason in decisions:
    print(f'{workload:<45} {engine:<10} {reason}')

print('\n---')
print('Rule of thumb: Start with vLLM. Switch to SGLang for structured/prefix workloads.')
print('Graduate to TRT-LLM when throughput justifies compilation investment.')